# Banking Customer & Credit Risk Analytics

**Project focus:** Loan default analysis, customer risk segmentation and predictive modelling  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, SQLite/SQL and Scikit-learn

## Project objective
Analyse historical loan records to understand patterns associated with loan defaults, explore customer and loan risk segments, and train a Logistic Regression model to estimate the likelihood of default.

> **Important:** This is an analytical portfolio project using historical data. The patterns found are associations, not proof of causation. A model prediction is an estimate, not a guarantee of what a borrower will do.

## Workflow
1. Load and inspect the dataset.
2. Clean missing values, duplicates and implausible age records.
3. Explore loan defaults and customer risk segments.
4. Query the cleaned data using SQL (SQLite).
5. Train and evaluate a Logistic Regression model.
6. Compare classification thresholds and inspect model coefficients.
7. Export cleaned data and analysis outputs for use in Power BI.

## Before you start
Run the notebook from top to bottom. When prompted, upload the original `credit_risk_dataset.csv` file or a ZIP file containing it. Do not upload the already-cleaned file as the original input.

## 1. Import libraries

In [ ]:
import os
import zipfile
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)

RANDOM_STATE = 42
OUTPUT_DIR = Path("credit_risk_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

## 2. Load the original dataset

Upload either:
- `credit_risk_dataset.csv`, or
- a ZIP file containing `credit_risk_dataset.csv`.

The code searches for the CSV after extraction and loads it into `df`.

In [ ]:
# Find a dataset already present in the working directory.
csv_candidates = list(Path(".").glob("**/credit_risk_dataset.csv"))

if not csv_candidates:
    try:
        from google.colab import files
        print("Upload the original credit_risk_dataset.csv file or its ZIP archive.")
        uploaded = files.upload()

        for uploaded_name in uploaded:
            uploaded_path = Path(uploaded_name)
            if uploaded_path.suffix.lower() == ".zip":
                extract_dir = Path("credit_risk_input")
                extract_dir.mkdir(exist_ok=True)
                with zipfile.ZipFile(uploaded_path, "r") as archive:
                    archive.extractall(extract_dir)
            elif uploaded_path.suffix.lower() == ".csv":
                pass
    except ImportError:
        print("Google Colab upload is unavailable. Place credit_risk_dataset.csv in this folder.")

csv_candidates = list(Path(".").glob("**/credit_risk_dataset.csv"))
if not csv_candidates:
    raise FileNotFoundError(
        "Could not find credit_risk_dataset.csv. Upload the original CSV or a ZIP containing it."
    )

dataset_path = csv_candidates[0]
df = pd.read_csv(dataset_path)

print("Dataset loaded from:", dataset_path)
print("Original dataset shape:", df.shape)
display(df.head())

## 3. Understand the dataset

Check the column names, data types, missing values, target distribution, duplicates and basic numerical statistics.

In [ ]:
print("DATASET INFORMATION")
print("-------------------")
df.info()

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values by column:")
display(df.isna().sum().to_frame("missing_values"))

print("\nLoan status counts:")
display(df["loan_status"].value_counts().rename(index={0: "Non-default", 1: "Default"}))

print("\nLoan status percentages:")
display((df["loan_status"].value_counts(normalize=True) * 100).round(2))

print("\nExact duplicate rows:", df.duplicated().sum())

print("\nNumerical summary:")
display(df.describe().round(2))

## 4. Clean the data

Cleaning decisions:
- Fill missing employment length and interest rate with their respective medians.
- Remove exact duplicate rows.
- Exclude records with borrower age above 100 from the analysis because these values were treated as implausible for this project.
- Keep the original `df` unchanged and create a separate cleaned analysis dataset.

In [ ]:
required_columns = {
    "person_age", "person_income", "person_home_ownership",
    "person_emp_length", "loan_intent", "loan_grade",
    "loan_amnt", "loan_int_rate", "loan_status",
    "loan_percent_income", "cb_person_default_on_file",
    "cb_person_cred_hist_length"
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"The dataset is missing expected columns: {sorted(missing_columns)}")

df_clean = df.copy()

# Convert expected numeric fields safely.
numeric_columns = [
    "person_age", "person_income", "person_emp_length",
    "loan_amnt", "loan_int_rate", "loan_status",
    "loan_percent_income", "cb_person_cred_hist_length"
]
for column in numeric_columns:
    df_clean[column] = pd.to_numeric(df_clean[column], errors="coerce")

# Fill the two missing-value columns using their medians.
for column in ["person_emp_length", "loan_int_rate"]:
    df_clean[column] = df_clean[column].fillna(df_clean[column].median())

duplicates_found = int(df_clean.duplicated().sum())
df_clean = df_clean.drop_duplicates().copy()

ages_over_100 = int((df_clean["person_age"] > 100).sum())
df_analysis = df_clean.loc[df_clean["person_age"] <= 100].copy()

print("Original records:", len(df))
print("Duplicate rows removed:", duplicates_found)
print("Records with age above 100 excluded:", ages_over_100)
print("Final analysis dataset shape:", df_analysis.shape)
print("Remaining missing values:", int(df_analysis.isna().sum().sum()))
print("Remaining duplicate rows:", int(df_analysis.duplicated().sum()))

## 5. Exploratory data analysis (EDA)

The target variable is `loan_status`:
- `0` = loan did not default
- `1` = loan defaulted

Start by checking the overall balance between the two outcomes.

In [ ]:
status_counts = df_analysis["loan_status"].value_counts().sort_index()
status_percentages = (df_analysis["loan_status"].value_counts(normalize=True).sort_index() * 100)

display(pd.DataFrame({
    "loan_status": ["Non-default (0)", "Default (1)"],
    "records": [status_counts.get(0, 0), status_counts.get(1, 0)],
    "percentage": [status_percentages.get(0, 0), status_percentages.get(1, 0)]
}).round(2))

plt.figure(figsize=(7, 4))
sns.countplot(data=df_analysis, x="loan_status")
plt.title("Distribution of Loan Status")
plt.xlabel("Loan status (0 = non-default, 1 = default)")
plt.ylabel("Number of loans")
plt.tight_layout()
plt.show()

### 5.1 Borrower income and age checks

In [ ]:
display(df_analysis[["person_age", "person_income", "loan_amnt"]].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df_analysis["person_income"], bins=50, ax=axes[0])
axes[0].set_title("Distribution of Annual Income")
axes[0].set_xlabel("Annual income")

sns.histplot(df_analysis["person_age"], bins=30, ax=axes[1])
axes[1].set_title("Distribution of Borrower Age")
axes[1].set_xlabel("Age")

plt.tight_layout()
plt.show()

### 5.2 Default rate by loan grade

A default rate is calculated as the average of `loan_status` multiplied by 100. The number of records in each grade is shown alongside the rate so that small groups can be interpreted cautiously.

In [ ]:
grade_analysis = (
    df_analysis.groupby("loan_grade")
    .agg(
        total_loans=("loan_status", "size"),
        defaulted_loans=("loan_status", "sum"),
        default_rate_pct=("loan_status", lambda values: values.mean() * 100)
    )
    .sort_values("default_rate_pct", ascending=False)
)
display(grade_analysis.round(2))

plt.figure(figsize=(8, 5))
sns.barplot(data=grade_analysis.reset_index(), x="loan_grade", y="default_rate_pct")
plt.title("Observed Loan Default Rate by Loan Grade")
plt.xlabel("Loan grade")
plt.ylabel("Default rate (%)")
plt.tight_layout()
plt.show()

### 5.3 Default rate by loan purpose

In [ ]:
purpose_analysis = (
    df_analysis.groupby("loan_intent")
    .agg(
        total_loans=("loan_status", "size"),
        default_rate_pct=("loan_status", lambda values: values.mean() * 100)
    )
    .sort_values("default_rate_pct", ascending=False)
)
display(purpose_analysis.round(2))

plt.figure(figsize=(10, 5))
sns.barplot(data=purpose_analysis.reset_index(), x="loan_intent", y="default_rate_pct")
plt.title("Observed Loan Default Rate by Loan Purpose")
plt.xlabel("Loan purpose")
plt.ylabel("Default rate (%)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### 5.4 Create income groups and compare default rates

Borrowers are split into four income groups using quartiles. These are relative groups within this dataset, not universal income bands.

In [ ]:
df_analysis["income_group"] = pd.qcut(
    df_analysis["person_income"],
    q=4,
    labels=["Low", "Lower-Middle", "Upper-Middle", "High"],
    duplicates="drop"
)

income_analysis = (
    df_analysis.groupby("income_group", observed=True)
    .agg(
        total_loans=("loan_status", "size"),
        default_rate_pct=("loan_status", lambda values: values.mean() * 100)
    )
)
display(income_analysis.round(2))

plt.figure(figsize=(8, 5))
sns.barplot(data=income_analysis.reset_index(), x="income_group", y="default_rate_pct")
plt.title("Observed Loan Default Rate by Income Group")
plt.xlabel("Income group")
plt.ylabel("Default rate (%)")
plt.tight_layout()
plt.show()

### 5.5 Create loan-burden groups and compare default rates

`loan_percent_income` represents the loan amount as a proportion of annual income. We group this measure to compare observed default rates across burden bands.

In [ ]:
df_analysis["loan_burden_group"] = pd.cut(
    df_analysis["loan_percent_income"],
    bins=[-np.inf, 0.10, 0.20, 0.30, np.inf],
    labels=["Up to 10%", "10–20%", "20–30%", "Above 30%"]
)

burden_analysis = (
    df_analysis.groupby("loan_burden_group", observed=True)
    .agg(
        total_loans=("loan_status", "size"),
        defaulted_loans=("loan_status", "sum"),
        default_rate_pct=("loan_status", lambda values: values.mean() * 100)
    )
)
display(burden_analysis.round(2))

plt.figure(figsize=(8, 5))
sns.barplot(data=burden_analysis.reset_index(), x="loan_burden_group", y="default_rate_pct")
plt.title("Observed Loan Default Rate by Loan-to-Income Group")
plt.xlabel("Loan burden group")
plt.ylabel("Default rate (%)")
plt.tight_layout()
plt.show()

### 5.6 Default rate by home ownership

In [ ]:
home_analysis = (
    df_analysis.groupby("person_home_ownership")
    .agg(
        total_loans=("loan_status", "size"),
        default_rate_pct=("loan_status", lambda values: values.mean() * 100)
    )
    .sort_values("default_rate_pct", ascending=False)
)
display(home_analysis.round(2))

## 6. SQL analysis using SQLite

The cleaned analysis dataset is loaded into a SQLite table called `loan_data`. The queries below calculate overall default rates and compare risk patterns across loan grades, purposes, income groups, loan-burden groups and home ownership.

In [ ]:
conn = sqlite3.connect(":memory:")
df_analysis.to_sql("loan_data", conn, if_exists="replace", index=False)

print("SQLite table created successfully.")
print("Rows in loan_data:", pd.read_sql_query("SELECT COUNT(*) AS total_rows FROM loan_data;", conn).iloc[0, 0])

### 6.1 SQL query – overall default rate

In [ ]:
sql_overall = '''
SELECT
    COUNT(*) AS total_loans,
    SUM(loan_status) AS defaulted_loans,
    ROUND(AVG(loan_status) * 100, 2) AS default_rate_pct
FROM loan_data;
'''
display(pd.read_sql_query(sql_overall, conn))

### 6.2 SQL query – default rate by loan grade

In [ ]:
sql_grade = '''
SELECT
    loan_grade,
    COUNT(*) AS total_loans,
    SUM(loan_status) AS defaulted_loans,
    ROUND(AVG(loan_status) * 100, 2) AS default_rate_pct
FROM loan_data
GROUP BY loan_grade
ORDER BY default_rate_pct DESC;
'''
display(pd.read_sql_query(sql_grade, conn))

### 6.3 SQL query – default rate by loan purpose

In [ ]:
sql_purpose = '''
SELECT
    loan_intent,
    COUNT(*) AS total_loans,
    SUM(loan_status) AS defaulted_loans,
    ROUND(AVG(loan_status) * 100, 2) AS default_rate_pct
FROM loan_data
GROUP BY loan_intent
ORDER BY default_rate_pct DESC;
'''
display(pd.read_sql_query(sql_purpose, conn))

### 6.4 SQL query – default rate by income group

In [ ]:
sql_income = '''
SELECT
    income_group,
    COUNT(*) AS total_loans,
    SUM(loan_status) AS defaulted_loans,
    ROUND(AVG(loan_status) * 100, 2) AS default_rate_pct
FROM loan_data
GROUP BY income_group
ORDER BY default_rate_pct DESC;
'''
display(pd.read_sql_query(sql_income, conn))

### 6.5 SQL query – default rate by loan-burden group

In [ ]:
sql_burden = '''
SELECT
    loan_burden_group,
    COUNT(*) AS total_loans,
    SUM(loan_status) AS defaulted_loans,
    ROUND(AVG(loan_status) * 100, 2) AS default_rate_pct
FROM loan_data
GROUP BY loan_burden_group
ORDER BY default_rate_pct DESC;
'''
display(pd.read_sql_query(sql_burden, conn))

### 6.6 SQL query – default rate by home ownership

In [ ]:
sql_home = '''
SELECT
    person_home_ownership,
    COUNT(*) AS total_loans,
    SUM(loan_status) AS defaulted_loans,
    ROUND(AVG(loan_status) * 100, 2) AS default_rate_pct
FROM loan_data
GROUP BY person_home_ownership
ORDER BY default_rate_pct DESC;
'''
display(pd.read_sql_query(sql_home, conn))

### 6.7 SQL query – combined income and loan-burden segments

The `HAVING` clause excludes very small groups with fewer than 30 records, reducing the risk of over-interpreting rates based on tiny samples.

In [ ]:
sql_segments = '''
SELECT
    income_group,
    loan_burden_group,
    COUNT(*) AS total_loans,
    SUM(loan_status) AS defaulted_loans,
    ROUND(AVG(loan_status) * 100, 2) AS default_rate_pct,
    CASE
        WHEN AVG(loan_status) >= 0.50 THEN 'Very high observed default rate'
        WHEN AVG(loan_status) >= 0.25 THEN 'High observed default rate'
        WHEN AVG(loan_status) >= 0.15 THEN 'Moderate observed default rate'
        ELSE 'Lower observed default rate'
    END AS risk_segment
FROM loan_data
GROUP BY income_group, loan_burden_group
HAVING COUNT(*) >= 30
ORDER BY default_rate_pct DESC;
'''
segment_results = pd.read_sql_query(sql_segments, conn)
display(segment_results)

## 7. Prepare data for machine learning

The target is `loan_status`:
- `0` = non-default
- `1` = default

The two descriptive groups created for analysis (`income_group` and `loan_burden_group`) are excluded from model inputs. The original numeric and categorical borrower/loan fields are used as predictors.

The preprocessing steps are fitted inside a Scikit-learn pipeline after the train/test split to avoid learning preprocessing information from the test data.

In [ ]:
df_ml = df_analysis.copy()

y = df_ml["loan_status"].astype(int)
X = df_ml.drop(columns=["loan_status", "income_group", "loan_burden_group"])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Training default rate:", round(y_train.mean() * 100, 2), "%")
print("Testing default rate:", round(y_test.mean() * 100, 2), "%")
print("Predictor columns:", X.columns.tolist())

## 8. Build and train the Logistic Regression model

Numerical variables are median-imputed and standardised. Categorical variables are imputed with their most frequent value and one-hot encoded. Logistic Regression is then trained on the processed training data.

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])

model.fit(X_train, y_train)
print("Logistic Regression model trained successfully.")
print("Numerical predictors:", len(numeric_features))
print("Categorical predictors:", len(categorical_features))

## 9. Evaluate model performance

The default classification threshold is 0.50. The evaluation includes accuracy, precision, recall, F1-score and ROC-AUC. The test set is held out from model training.

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

metrics_df = pd.DataFrame(
    [{"Metric": key, "Value": value} for key, value in metrics.items()]
)
display(metrics_df.round(4))

print("Classification report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Non-default", "Default"],
    zero_division=0
))

### 9.1 Confusion matrix and ROC curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["Non-default", "Default"],
    cmap="Blues",
    ax=axes[0],
    colorbar=False
)
axes[0].set_title("Confusion Matrix")

RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1])
axes[1].set_title("ROC Curve")

plt.tight_layout()
plt.show()

## 10. Compare classification thresholds

A threshold changes how readily the model labels a loan as a predicted default. A lower threshold generally identifies more actual defaults (higher recall), but may also create more false positives (lower precision).

The table below calculates precision, recall and F1-score at thresholds 0.30, 0.40, 0.50 and 0.60 using the same test-set probabilities.

In [ ]:
threshold_rows = []

for threshold in [0.30, 0.40, 0.50, 0.60]:
    threshold_prediction = (y_prob >= threshold).astype(int)
    threshold_rows.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, threshold_prediction, zero_division=0),
        "Recall": recall_score(y_test, threshold_prediction, zero_division=0),
        "F1-score": f1_score(y_test, threshold_prediction, zero_division=0)
    })

threshold_results = pd.DataFrame(threshold_rows)
display(threshold_results.round(4))

### How to interpret the threshold comparison

- A lower threshold such as 0.30 is more likely to flag a loan as a potential default. This can increase recall, but it can also flag more borrowers who would not default.
- A higher threshold such as 0.60 is more selective. It can improve precision while missing more actual defaults.
- Choose a threshold based on the business objective and the cost of false positives versus missed defaults. Do not choose a threshold using accuracy alone.

## 11. Inspect Logistic Regression coefficients

The model coefficients show how each encoded feature is associated with the model's predicted log-odds of default, holding the other included features constant. Positive coefficients increase the model's estimated default odds; negative coefficients decrease them.

These coefficients are model associations, not proof of causation or standalone measures of business importance.

In [ ]:
fitted_preprocessor = model.named_steps["preprocessor"]
fitted_classifier = model.named_steps["classifier"]

feature_names = fitted_preprocessor.get_feature_names_out()
coefficients = fitted_classifier.coef_[0]

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})
feature_importance["Absolute_Importance"] = feature_importance["Coefficient"].abs()
feature_importance = feature_importance.sort_values(
    "Absolute_Importance", ascending=False
)

display(feature_importance.head(15).round(4))

top_features = feature_importance.head(10).sort_values("Coefficient")
plt.figure(figsize=(10, 6))
plt.barh(top_features["Feature"], top_features["Coefficient"])
plt.axvline(0, linewidth=1)
plt.title("Top 10 Logistic Regression Coefficients")
plt.xlabel("Coefficient (direction and strength in the fitted model)")
plt.ylabel("Encoded feature")
plt.tight_layout()
plt.show()

## 12. Key findings and business interpretation

The following tables are calculated from the cleaned dataset. Use them to identify the groups with the highest observed default rates, while also checking the number of records in each group.

Potential business implications:
- Review higher-risk loan grades more carefully, while checking the size of each grade group.
- Consider affordability measures such as loan amount relative to borrower income.
- Use customer and loan characteristics together rather than relying on one variable.
- Monitor model performance and fairness before any real-world lending use.

**Caution:** The dataset is historical and may not represent every bank or current lending population. Differences between groups do not establish that a characteristic causes default. This model is a learning/portfolio exercise, not a production credit decision system.

In [ ]:
print("Overall default rate:")
print(round(df_analysis["loan_status"].mean() * 100, 2), "%")

print("\nDefault rate by loan grade:")
display(grade_analysis.round(2))

print("\nDefault rate by income group:")
display(income_analysis.round(2))

print("\nDefault rate by loan-burden group:")
display(burden_analysis.round(2))

print("\nDefault rate by home ownership:")
display(home_analysis.round(2))

## 13. Export project outputs

This cell saves the cleaned dataset, model metrics, threshold comparison, coefficient table, segment analysis and SQL queries. These files can be used when building or updating the Power BI dashboard and when documenting the GitHub repository.

In [ ]:
# Save analysis tables.
df_analysis.to_csv(OUTPUT_DIR / "credit_risk_cleaned.csv", index=False)
metrics_df.to_csv(OUTPUT_DIR / "credit_risk_model_metrics.csv", index=False)
threshold_results.to_csv(OUTPUT_DIR / "credit_risk_threshold_comparison.csv", index=False)
feature_importance.to_csv(OUTPUT_DIR / "credit_risk_feature_importance.csv", index=False)
segment_results.to_csv(OUTPUT_DIR / "credit_risk_segments.csv", index=False)

# Combine the SQL queries into a standalone .sql file.
all_sql = "\n\n".join([
    "-- Overall default rate\n" + sql_overall,
    "-- Default rate by loan grade\n" + sql_grade,
    "-- Default rate by loan purpose\n" + sql_purpose,
    "-- Default rate by income group\n" + sql_income,
    "-- Default rate by loan-burden group\n" + sql_burden,
    "-- Default rate by home ownership\n" + sql_home,
    "-- Combined income and loan-burden segments\n" + sql_segments
])

sql_file = OUTPUT_DIR / "credit_risk_analysis.sql"
sql_file.write_text(all_sql, encoding="utf-8")

print("Files saved in:", OUTPUT_DIR.resolve())
for output_file in sorted(OUTPUT_DIR.iterdir()):
    print("-", output_file.name, f"({output_file.stat().st_size:,} bytes)")

## 14. Final project summary

This project used Python to clean and explore loan records, SQL to query default patterns, and Scikit-learn to train and evaluate a Logistic Regression model. The threshold comparison demonstrated the trade-off between precision and recall. The analysis outputs can be presented through an interactive Power BI dashboard.

### Skills demonstrated
- Data cleaning and preparation with Pandas
- Exploratory data analysis and visualisation
- SQL aggregation, grouping, filtering and segmentation
- Classification modelling with Logistic Regression
- Model evaluation using accuracy, precision, recall, F1-score and ROC-AUC
- Communicating analytical findings and business implications

### Power BI dashboard
A separate Power BI dashboard was created for this project. It contains three pages, three KPI cards and six visualisations. Add the `.pbix` file and dashboard screenshots to the GitHub repository separately if you have them saved.

### Reproducibility note
Run all notebook cells from top to bottom after uploading the original dataset. The model metrics are calculated from the current run rather than being typed into the notebook as fixed results.